# E.R.M.E.S. - Fase 3: Valutazione Offline e Modelli Baseline

Prima di introdurre la complessità architetturale del Deep Learning, è fondamentale stabilire dei termini di paragone oggettivi. In questo notebook implementiamo diverse **Baseline** (modelli di riferimento) per definire le soglie minime di performance:

1. **Zero Rule Baseline:** Un'euristica banale che predice sempre la classe maggioritaria del dataset. Serve a dimostrare la gravità dello sbilanciamento (*Class Imbalance*).
2. **Random Baseline:** Predizioni generate casualmente seguendo la distribuzione stratificata delle etichette.
3. **Soluzioni Esistenti (Algorithmic Baselines):** Classificatori di Machine Learning classico per testare i limiti delle *engineered features*:
   - **SVM Lineare + PCA:** Per testare la separabilità lineare nello spazio compresso.
   - **Random Forest (Ensemble):** Per sfruttare la logica del *Bagging* (media di più alberi decisionali) e catturare pattern non lineari.
4. **Human Baseline (Riferimento Teorico):** Nel task FER-2013, la letteratura stima l'accuratezza umana intorno al **65%**. Questo rappresenta il nostro limite superiore (*Upper Bound*).

In [ ]:
"""
Configurazione dell'ambiente e importazione dei moduli Scikit-Learn
per il Machine Learning classico e le Naive Baselines.
"""

import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Import dei Modelli
from sklearn.dummy import DummyClassifier
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from typing import Tuple

import logging
tf.get_logger().setLevel(logging.ERROR)

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_context("notebook", font_scale=1.1)

print("--- INIZIALIZZAZIONE AMBIENTE MACHINE LEARNING ---")

## 3.1 Estrazione e Vettorializzazione dei Dati (Flattening)
Gli algoritmi di Machine Learning classico non supportano tensori bidimensionali (matrici di pixel). Dobbiamo eseguire il *Flattening*: ogni immagine $48 \times 48$ verrà "srotolata" in un singolo vettore monodimensionale $X \in \mathbb{R}^{2304}$. Questa operazione, purtroppo, distrugge completamente la gerarchia spaziale del volto.

In [ ]:
def extract_and_flatten(dataset_dir: Path) -> Tuple[np.ndarray, np.ndarray]:
    """
    Carica le immagini direttamente dal disco, le converte in Grayscale e 
    ne esegue il flattening per renderle compatibili con Scikit-Learn.
    
    Args:
        dataset_dir (Path): Cartella root del dataset (train o test).
        
    Returns:
        Tuple[np.ndarray, np.ndarray]: Matrice delle feature (X) e vettore etichette (y).
    """
    dataset = tf.keras.utils.image_dataset_from_directory(
        dataset_dir,
        color_mode='grayscale',
        image_size=(48, 48),
        batch_size=1000, # Batch larghi per velocizzare l'estrazione in memoria RAM
        shuffle=False,
        label_mode='int'
    )
    
    X_list, y_list = [], []
    for images, labels in dataset:
        # Flattening: da (Batch, 48, 48, 1) a (Batch, 2304)
        flat_images = tf.reshape(images, (images.shape[0], -1)).numpy()
        
        # Normalizzazione Min-Max manuale [0, 1]
        flat_images = flat_images / 255.0 
        
        X_list.append(flat_images)
        y_list.append(labels.numpy())
        
    return np.vstack(X_list), np.concatenate(y_list)

print("Estrazione e Flattening del Training Set in corso...")
X_train, y_train = extract_and_flatten(Path('../data/fer2013/train'))

print("Estrazione e Flattening del Test Set in corso...")
X_test, y_test = extract_and_flatten(Path('../data/fer2013/test'))

print(f"\nDimensioni Finali -> X_train: {X_train.shape}, y_train: {y_train.shape}")

## 3.2 Implementazione Naive Baselines
Costruiamo i modelli "stupidi" che ci serviranno come pavimento prestazionale. 
* Il **Zero Rule** ignora i dati in input e predice sempre "Happy" (la classe più frequente identificata nell'EDA). 
* Il **Random Classifier** lancia un dado truccato basato sulla distribuzione statistica delle classi.

In [ ]:
print("--- ADDESTRAMENTO NAIVE BASELINES ---")

# 1. Zero Rule Baseline (Prevede sempre la classe maggioritaria)
zero_rule_model = DummyClassifier(strategy='most_frequent')
zero_rule_model.fit(X_train, y_train)
y_pred_zero = zero_rule_model.predict(X_test)
f1_zero = f1_score(y_test, y_pred_zero, average='macro')
print(f"Zero Rule Baseline -> Macro F1-Score: {f1_zero:.4f}")

# 2. Random Baseline (Predizione stocastica stratificata)
random_model = DummyClassifier(strategy='stratified', random_state=42)
random_model.fit(X_train, y_train)
y_pred_random = random_model.predict(X_test)
f1_random = f1_score(y_test, y_pred_random, average='macro')
print(f"Random Baseline    -> Macro F1-Score: {f1_random:.4f}")

## 3.3 Soluzioni Esistenti (Algorithmic Baselines)

In questa sezione addestriamo due algoritmi classici per sondare limiti diversi:
1. **Support Vector Machine (SVM):** Adottiamo una pipeline che integra la **PCA** (preservando il 95% della varianza spiegata) per la riduzione dimensionale, unita a un classificatore SVM lineare con pesi bilanciati.
2. **Random Forest (Ensemble):** Algoritmo basato su alberi decisionali (*Bagging*). Essendo robusto all'alta dimensionalità, lo addestriamo direttamente sui 2304 pixel senza PCA, permettendogli di esplorare relazioni non lineari.

In [ ]:
"""
Addestramento Sequenziale dei Modelli Classici.
Entrambi i modelli utilizzano 'class_weight=balanced' per contrastare il bias.
"""

# --- 1. PIPELINE SVM ---
svm_pipeline = Pipeline([
    ('pca', PCA(n_components=0.95, random_state=42)),
    ('classifier', LinearSVC(class_weight='balanced', dual=False, max_iter=2000, random_state=42))
])

print("--- ADDESTRAMENTO SVM (MODELLO LINEARE) ---")
print("Riduzione dimensionale e ottimizzazione iperpiani in corso...")
svm_pipeline.fit(X_train, y_train)
y_pred_svm = svm_pipeline.predict(X_test)
f1_svm = f1_score(y_test, y_pred_svm, average='macro')
print(f"SVM Baseline -> Macro F1-Score: {f1_svm:.4f}\n")

# --- 2. RANDOM FOREST (ENSEMBLE) ---
print("--- ADDESTRAMENTO RANDOM FOREST (ENSEMBLE) ---")
print("Costruzione degli alberi decisionali in corso...")
# n_jobs=-1 usa tutti i core della CPU per velocizzare l'addestramento
rf_model = RandomForestClassifier(n_estimators=150, class_weight='balanced', random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)
f1_rf = f1_score(y_test, y_pred_rf, average='macro')
print(f"Random Forest Baseline -> Macro F1-Score: {f1_rf:.4f}")

In [ ]:
"""
Cruscotto di Valutazione Finale Esteso.
Mettiamo a confronto visivo le Baselines e analizziamo le Matrici di Confusione 
sia del modello lineare (SVM) che dell'Ensemble (Random Forest).
"""

emotions = ['Angry', 'Disgust', 'Fear', 'Happy', 'Neutral', 'Sad', 'Surprise']

fig, axes = plt.subplots(1, 3, figsize=(24, 7), gridspec_kw={'width_ratios': [1, 1, 1.2]})
fig.suptitle('Analisi Prestazionale: Naive vs Machine Learning Classico', fontsize=18, fontweight='bold', y=1.05)

# 1. Matrice di Confusione della SVM
cm_svm = confusion_matrix(y_test, y_pred_svm)
sns.heatmap(cm_svm, annot=True, fmt='d', cmap='Reds', 
            xticklabels=emotions, yticklabels=emotions, 
            annot_kws={"size": 11}, ax=axes[0])
axes[0].set_title('Matrice di Confusione - SVM\n(Linear + PCA)', fontsize=14, pad=10)
axes[0].set_ylabel('Classe Reale (True)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Classe Predetta (Predicted)', fontsize=12, fontweight='bold')

# 2. Matrice di Confusione della Random Forest
cm_rf = confusion_matrix(y_test, y_pred_rf)
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Greens', 
            xticklabels=emotions, yticklabels=emotions, 
            annot_kws={"size": 11}, ax=axes[1])
axes[1].set_title('Matrice di Confusione - Random Forest\n(Ensemble Non-Lineare)', fontsize=14, pad=10)
axes[1].set_ylabel('Classe Reale (True)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Classe Predetta (Predicted)', fontsize=12, fontweight='bold')

# 3. Benchmarking di tutte le Baselines
baseline_names = ['Zero Rule\n(Major Class)', 'Random\n(Stratified)', 'SVM\n(Linear+PCA)', 'Random Forest\n(Ensemble)']
baseline_scores = [f1_zero, f1_random, f1_svm, f1_rf]

sns.barplot(x=baseline_names, y=baseline_scores, hue=baseline_names, palette='magma', ax=axes[2], legend=False)
axes[2].set_title('Confronto Macro F1-Score Globale', fontsize=14, pad=10)
axes[2].set_ylabel('Macro F1-Score', fontsize=12, fontweight='bold')
axes[2].set_ylim(0, 0.70) # Limite fissato sul theoretical umano (65%)

# Linea tratteggiata per la Human Baseline
axes[2].axhline(y=0.65, color='green', linestyle='--', linewidth=2.5, alpha=0.8)
axes[2].text(1.5, 0.66, 'Human Baseline (~0.65)', color='green', fontsize=12, fontweight='bold', ha='center')

# Valori numerici sulle barre
for i, v in enumerate(baseline_scores):
    axes[2].text(i, v + 0.01, f"{v:.2f}", color='black', ha='center', fontweight='bold', fontsize=12)

plt.tight_layout()
plt.savefig('baseline_benchmarks.svg', format='svg', bbox_inches='tight')
plt.show()

print("\n--- REPORT: SVM (MODELLO LINEARE) ---")
print(classification_report(y_test, y_pred_svm, target_names=emotions))

print("\n--- REPORT: RANDOM FOREST (ENSEMBLE NON-LINEARE) ---")
print(classification_report(y_test, y_pred_rf, target_names=emotions))

## Conclusioni della Fase 3

L'analisi comparativa dei modelli di Machine Learning classico applicati alla Computer Vision permette di delineare una gerarchia di efficacia basata sulla complessità computazionale e sulla natura dei classificatori:

1. **Limiti della separabilità lineare (SVM):** Il modello Support Vector Machine (SVM) ha registrato prestazioni ridotte, con un **F1-Score del 28%**. Nonostante l'applicazione della Principal Component Analysis (PCA) per la riduzione della dimensionalità, la ricerca di iperpiani di separazione lineari si è dimostrata inadeguata alla complessità del dataset. Tale limite è confermato dall'incapacità del modello di generalizzare correttamente le classi minoritarie, dove l'uso dei *class weights* ha prodotto una Precision del 7% (classe "Disgust"), indicando una classificazione prossima alla distribuzione casuale.

2. **Efficacia degli approcci non lineari (Random Forest):** L'impiego di una metodologia *Ensemble* basata su 150 alberi decisionali ha prodotto un incremento significativo delle prestazioni, raggiungendo un **F1-Score del 44%**. La Random Forest ha dimostrato una superiore capacità nel modellare relazioni non lineari tra le 2304 feature estratte, gestendo l'alta dimensionalità dello spazio delle caratteristiche in modo più efficiente rispetto alla SVM e senza la necessità di pre-elaborazione tramite PCA.

3. **Criticità del Flattening e perdita dell'informazione spaziale:** Nonostante i miglioramenti ottenuti con le tecniche di Bagging, il valore di F1-Score (44%) rimane sensibilmente inferiore alla **Human Baseline (65%)**. I dati suggeriscono che il limite intrinseco dei modelli classici risieda nell'operazione di *flattening*: la vettorializzazione dell'input distrugge la gerarchia spaziale e le relazioni topologiche tra i pixel (come le distanze relative tra i landmark facciali).

In conclusione, l'impossibilità di superare i limiti prestazionali descritti senza preservare la struttura bidimensionale dell'immagine rende necessario, per la successiva Fase 4, il passaggio a paradigmi di Deep Learning. L'adozione di reti neurali convoluzionali (CNN) consentirà infatti l'estrazione automatica di feature gerarchiche tramite filtri spaziali, superando i vincoli della classificazione basata esclusivamente su vettori appiattiti.